In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")
import decoupler as dc
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
sc.settings.set_figure_params(dpi=150, facecolor="white")
sc.settings.figdir = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\in_depth"
figdir = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\in_depth"
leiden_resolution = 0.2

In [ ]:
cluster_colourmap = {
    # "0": "#F5C423",  # amber
    # "1": "#5B5DE6",  # indigo
    "3": "#E16ED1",  # magenta
    "2": "#2E9BD3",  # azure
    # "2": "#8B3FD1",  # violet
    "0": "#F06B2D",  # orange (instead of red)
    "1": "#C32A71",  # raspberry
    # "7": "#238F42",  # green
    # "8": "#A7C83A",  # lime
}
# condition_mapping = {"Bead" : "Vasculature", 
#                      "FL_Chip_minus" : "iFL_V+", 
#                      "FL_Chip_plus" : "tFL_V+",
#                        "FL_ULA_minus" : "iFL_V-", 
#                        "FL_ULA_plus" : "tFL_V-",
#                        "rLN_Chip_minus" : "irLN_V+",
#                        "rLN_ULA_minus" : "irLN_V-",}
condition_colormap = {"AAG05734_WP_Yellow": "#E8EB5E", 
                      "AAG05733_WW_Red": "#C51BCE", 
                      "AAG05732_WS_Blue": "#2737FB",
                      "AAG05735_PP_Green": "#5B9514",
}


clean_matrix = r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Output\adata_clustered_res_0_2.h5ad"
res_0_2 = sc.read_h5ad(clean_matrix)
# res_0_2.obs["condition"] = res_0_2.obs["condition"].replace(condition_mapping, inplace=False)
res_0_2.obs["condition"].unique()

In [ ]:
# scanpy's `palette` applies a single palette to ALL panels, so it can't take a
# list of one palette per color. Instead, assign each categorical's colours into
# .uns (ordered to match its categories) and let umap pick them up automatically.
res_0_2.obs["clusters"] = res_0_2.obs["clusters"].astype("category")
res_0_2.obs["condition"] = res_0_2.obs["condition"].astype("category")
res_0_2.obs["sample"] = res_0_2.obs["sample"].astype("category")

res_0_2.uns["clusters_colors"] = [cluster_colourmap[c] for c in res_0_2.obs["clusters"].cat.categories]
res_0_2.uns["condition_colors"] = [condition_colormap[c] for c in res_0_2.obs["condition"].cat.categories]
res_0_2.uns["sample_colors"] = [
    plt.matplotlib.colors.to_hex(c)
    for c in plt.get_cmap("tab20").colors[:res_0_2.obs["sample"].cat.categories.size]
]

sc.pl.umap(res_0_2, color=["clusters", "condition"], wspace=0.4, ncols=2,
           save="_overview.png", show=True)

In [ ]:
def highlight_umap(adata, groupby, size=10, ncols=4, na_color="lightgrey", savefig=None, palette = None):
    """Highlight one category at a time on the full UMAP (others greyed out)."""
    groups = list(adata.obs[groupby].cat.categories) if hasattr(adata.obs[groupby], "cat") \
        else sorted(adata.obs[groupby].unique())
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(4 * ncols, 4 * nrows),
        gridspec_kw={"wspace": 0.3, "hspace": 0.3},
    )
    axes = np.atleast_1d(axes).ravel()
    for i, (ax, group) in enumerate(zip(axes, groups)):
        sc.pl.umap(
            adata, color=groupby, groups=[group], na_color=na_color,
            size=size, title=str(group), legend_loc=None, frameon=False, show=False, ax=ax, palette=[palette[group]] if palette else None,    
        )
    for ax in axes[len(groups):]:
        ax.axis("off")
    if savefig:
        plt.savefig(savefig, bbox_inches="tight")
    plt.show()


highlight_umap(res_0_2, "condition", size=10, ncols=4,
               savefig=os.path.join(figdir, "umap_condition_highlight.png"), palette=condition_colormap)

In [ ]:
comp = pd.crosstab(res_0_2.obs["condition"], res_0_2.obs["clusters"], normalize="index")
ax = comp.plot(kind="bar", stacked=True, figsize=(8, 4),  color=[cluster_colourmap[str(c)] for c in comp.columns], width=0.85, grid = False)
ax.set_ylabel("Fraction of cells")
ax.set_xlabel("Condition")
ax.legend(title="cluster", bbox_to_anchor=(1.01, 1), loc="upper left", ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(figdir, "composition_per_condition.png"), bbox_inches="tight")
plt.show()
comp.round(3).to_csv(os.path.join(figdir, "composition_per_condition.csv"))

In [ ]:
sc.tl.rank_genes_groups(res_0_2, groupby="clusters", method="wilcoxon", use_raw=True, rankby_abs = True)
sc.pl.rank_genes_groups_dotplot( #plots the top 5 DEGs per cluster (by p value), coloured by their log fold change
    res_0_2, n_genes=5, values_to_plot="logfoldchanges", cmap="bwr",
    vmin=-4, vmax=4, colorbar_title="log fold change",
    save="_markers.png", show=True,
)

In [ ]:
# # Canonical human immune markers grouped by lineage (lymph node / lymphoma context).
# immune_markers = {
#     "B cell": ["CD19", "MS4A1", "CD79A", "CD79B"],
#     "Germinal center B": ["BCL6", "AICDA", "RGS13", "LMO2"],
#     "FL B cell": ["BCL2", "MME"],          
#     "Plasma": ["MZB1", "XBP1", "PRDM1", "SDC1"],
#     "T cell": ["CD3D", "CD3E", "CD2"],
#     "CD4 T": ["CD4", "IL7R"],
#     "CD8 T": ["CD8A", "CD8B", "GZMK"],
#     "Tfh": ["CXCL13", "PDCD1", "ICOS"],
#     "NK": ["NKG7", "GNLY", "KLRD1"],
#     "Myeloid/DC": ["LYZ", "CD68", "ITGAX"],
#     "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5"],          # blood vessels
#     "Lymphatic endothelial": ["PROX1", "LYVE1", "PDPN"],       # lymphatic sinuses
#     "Fibroblast/stromal": ["PDGFRA", "PDGFRB", "COL1A1", "COL3A1", "DCN", "LUM"],
#     "FRC": ["ACTA2", "PDPN", "CCL19", "CCL21"],                 # fibroblastic reticular cells
#     "FDC": ["CR2", "CR1", "FDCSP", "CLU"],                      # follicular dendritic cells (germinal centers)
#     "Proliferating": ["MKI67", "TOP2A"],
# }

# # 5. Canonical marker annotation (drop genes absent from this dataset).
# present = {k: [g for g in v if g in res_0_2.var_names] for k, v in immune_markers.items()}
# present = {k: v for k, v in present.items() if v}

# # Recompute the dendrogram for THIS resolution's clusters. It is cached in
# # adata.uns and would otherwise be stale (wrong # categories) from a prior resolution.
# sc.tl.dendrogram(res_0_2, groupby="clusters")
# sc.pl.dotplot(res_0_2, present, groupby="clusters", standard_scale="var",
#                 dendrogram=True, save="_canonical_markers.png", show=True)


In [ ]:
# individual_genes = pd.read_excel(r"z:\Bel\Marina_Side_Projects\scRNA\SIGNATURE PDLT.xlsx")

# for gene in individual_genes.Gene:
#     if gene not in res_0_2.var_names:
#         print(f"Skipping {gene} — not in dataset")
#         continue

#     df = sc.get.obs_df(res_0_2, keys=[gene, "clusters"], use_raw=True)

#     summary = (
#         df.groupby("clusters")[gene]
#           .agg(
#               mean_expr="mean",
#               pct_expr=lambda x: (x > 0).mean() * 100,
#               n_cells="size"
#           )
#           .sort_values("mean_expr", ascending=False)
#     )


#     individual_genes.loc[individual_genes.Gene == gene, "Actual_Cluster"] = int(summary.index[0])



In [ ]:
# individual_genes["Actual_Cluster"] = individual_genes["Actual_Cluster"].astype("Int64")  
# individual_genes["Flag"] = individual_genes.apply(lambda row: True if row["Actual_Cluster"] != row["Predicted_Cluster"] else False, axis=1)

# individual_genes.to_csv(r"individual_gene_clusters_annotated.csv", index=False)
# individual_genes

In [ ]:
res_0_2


In [ ]:
res_0_2.obs.condition.unique()

In [ ]:
res_0_2[res_0_2.obs.condition == "AAG05734_WP_Yellow"]

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

gene_dz_excel = pd.read_excel(r"c:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Genes_DZ.xlsx")
gene_cc_excel = pd.read_excel(r"C:\Users\taylorhearn\Desktop\scRNAseq_Dorota\Genes_from_Cellchatt.xlsx")
all_genes = pd.concat([gene_dz_excel, gene_cc_excel], ignore_index=True)
key_genes = [g for g in all_genes.Gene if g in res_0_2.var_names]

conditions = list(res_0_2.obs.condition.cat.categories)
cmap = "Reds"

for gene in key_genes:
    expr = sc.get.obs_df(res_0_2, keys=[gene], use_raw=True)[gene]
    vmin, vmax = 0.0, float(expr.max())

    fig, axes = plt.subplots(ncols=4, figsize=(4 * 4, 4))
    axes = np.atleast_1d(axes).ravel()
    for ax, condition in zip(axes, conditions):
        sc.pl.umap(
            res_0_2[res_0_2.obs.condition == condition], color=gene,
            cmap=cmap, use_raw=True, vmin=vmin, vmax=vmax,
            colorbar_loc=None,               # suppress per-panel colorbars
            title=str(condition), frameon=False, show=False, ax=ax,
        )

    # One shared colorbar for the whole figure.
    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    fig.colorbar(sm, ax=axes.tolist(), fraction=0.02, pad=0.02, label=gene)
    fig.suptitle(gene, y=1.02, x=0.55)
    fig.savefig(os.path.join(figdir, f"umap_{gene}_by_condition.png"), bbox_inches="tight")
    plt.show()

In [ ]:
for gene in key_genes:
    sc.pl.violin(res_0_2, keys=gene, groupby="clusters", use_raw=True, stripplot=False, palette=cluster_colourmap, save=f"_{gene}_violin.png", show=False)


# Gene Signatures

In [ ]:
signature_df = pd.read_csv("marina_gene_signatures.csv")

# When use_raw=True, scoring happens against res_0_2.raw — filter against that.
gene_universe = res_0_2.raw.var_names if res_0_2.raw is not None else res_0_2.var_names

for sig in signature_df["signature"].unique():
    up = signature_df.query("signature == @sig and direction == 'up'")["gene"]
    down = signature_df.query("signature == @sig and direction == 'down'")["gene"]

    up = [g for g in up if g in gene_universe]
    down = [g for g in down if g in gene_universe]
    print(f"{sig}: {len(up)} up genes, {len(down)} down genes found")

    if not up and not down:
        print(f"  Skipping {sig} — no signature genes in dataset")
        continue

    if up:
        sc.tl.score_genes(res_0_2, up, score_name=f"{sig}_up", use_raw=True)
    if down:
        sc.tl.score_genes(res_0_2, down, score_name=f"{sig}_down", use_raw=True)

    # signed score (handle the case where one direction is empty)
    up_score = res_0_2.obs[f"{sig}_up"] if up else 0
    down_score = res_0_2.obs[f"{sig}_down"] if down else 0
    res_0_2.obs[f"{sig}_signed"] = up_score - down_score

    vmax = res_0_2.obs[f"{sig}_signed"].abs().quantile(0.99)
    sc.pl.umap(
        res_0_2, color=f"{sig}_signed", cmap="RdBu_r",
        vmin=-vmax, vcenter=0, vmax=vmax,
        title=f"{sig} signature (up − down)",
        save=f"_{sig}_signature.png", show=True,
    )

# Volcano Plots

In [ ]:
import re
from pydeseq2.default_inference import DefaultInference


def _safe_level(name):
    """pydeseq2 can't handle  '+', '-', or '_' 
     Conver to safe names e.g. 'iFL_V+' -> 'iFLVpos'."""
    return re.sub(r"[^0-9A-Za-z]", "", name.replace("+", "pos").replace("-", "neg"))


def pseudobulk_de(adata, cluster, cond_a, cond_b, sample_key="sample",
                  cond_key="condition", cluster_key="clusters",
                  layer="counts", min_cells=10):
    """Pseudobulk DE (cond_a vs cond_b) within one cluster. Positive log2FC = up in cond_a."""
    mask = (adata.obs[cluster_key] == cluster) & adata.obs[cond_key].isin([cond_a, cond_b])
    sub = adata[mask].copy()

    pb = dc.get_pseudobulk(
        sub, sample_col=sample_key, groups_col=None,
        layer=layer, mode="sum", min_cells=min_cells, min_counts=1000,
    )

    n = pb.obs[cond_key].value_counts()
    if n.get(cond_a, 0) < 2 or n.get(cond_b, 0) < 2:
        print(f"cluster {cluster}: not enough replicates "
              f"({cond_a}={n.get(cond_a,0)}, {cond_b}={n.get(cond_b,0)}) — skipping")
        return None

    sc.pp.filter_genes(pb, min_cells=1)

    # Overwrite "condition" with alphanumeric-only level names so pydeseq2 leaves them
    # untouched; astype(str) also drops the 5 unused categories from the full dataset.
    sa, sb = _safe_level(cond_a), _safe_level(cond_b)
    pb.obs[cond_key] = pb.obs[cond_key].astype(str).map({cond_a: sa, cond_b: sb}).astype("category")

    inference = DefaultInference(n_cpus=1)
    dds = DeseqDataSet(adata=pb, design_factors=cond_key, refit_cooks=True,
                       inference=inference)
    dds.deseq2()
    stat = DeseqStats(dds, contrast=[cond_key, sa, sb], inference=inference)  # cond_a vs cond_b
    stat.summary()
    res = stat.results_df.reset_index().rename(columns={"index": "gene"})
    res["cluster"] = cluster
    return res

In [ ]:
def volcano(df, cond_a, cond_b, cluster, lfc_thresh=1.0, padj_thresh=0.05, save=None):
    df = df.dropna(subset=["padj"]).copy()
    df["neglog10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))
    up   = (df["log2FoldChange"] >=  lfc_thresh) & (df["padj"] < padj_thresh)
    down = (df["log2FoldChange"] <= -lfc_thresh) & (df["padj"] < padj_thresh)

    plt.figure(figsize=(6, 5))
    plt.scatter(df["log2FoldChange"], df["neglog10_padj"], s=6, c="lightgrey")
    plt.scatter(df.loc[up, "log2FoldChange"],   df.loc[up, "neglog10_padj"],   s=10, c="red",  label=f"up in {cond_a}")
    plt.scatter(df.loc[down, "log2FoldChange"], df.loc[down, "neglog10_padj"], s=10, c="blue", label=f"up in {cond_b}")
    plt.axhline(-np.log10(padj_thresh), ls="--", c="k", lw=0.6)
    plt.axvline( lfc_thresh, ls="--", c="k", lw=0.6)
    plt.axvline(-lfc_thresh, ls="--", c="k", lw=0.6)
    plt.xlabel("log2 fold change"); plt.ylabel("-log10 adjusted p-value")
    plt.title(f"Cluster {cluster}: {cond_a} vs {cond_b}")
    plt.legend(fontsize=8)
    if save:
        plt.savefig(save, bbox_inches="tight")
    plt.show()


from itertools import combinations

# All possible condition pairs (every condition vs every other condition).
conditions = list(res_0_2.obs["condition"].cat.categories)
pairs = list(combinations(conditions, 2))


all_results = []
for cluster in res_0_2.obs["clusters"].cat.categories[5:]:
    for cond_a, cond_b in pairs:
        res = pseudobulk_de(res_0_2, cluster, cond_a, cond_b)
        if res is not None:
            all_results.append(res)
            volcano(res, cond_a, cond_b, cluster,
                    save=os.path.join(figdir, f"volcano_c{cluster}_{cond_a}_vs_{cond_b}.png"))

if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(os.path.join(figdir, "pseudobulk_DE_all.csv"), index=False)

In [ ]:
res_0_2